# Aligning STARmap to the Allen Brain Atlas (squidpy + STalign)

In this notebook, we align a STARmap dataset of the right hemisphere of a mouse brain to the Allen Brain Atlas.

**Note:** 3D atlas-to-2D-slice alignment (`LDDMM_3D_to_slice`) is not yet available in the squidpy high-level API. This notebook uses STalign functions directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nrrd
import torch
import squidpy as sq

from squidpy.experimental._lddmm import rasterize
from STalign import STalign

## Load STARmap data

In [ ]:
df = pd.read_csv('../starmap_data/well11_spatial.csv.gz')
x = np.array(df['X'])[1:].astype(float)
y = np.array(df['Y'])[1:].astype(float)

dx = 50
blur = 1

## Load Allen Brain Atlas

In [ ]:
url = 'http://api.brain-map.org/api/v2/data/query.csv?criteria=model::Structure,rma::criteria,[ontology_id$eq1],rma::options[order$eq%27structures.graph_order%27][num_rows$eqall]'
ontology_name, namesdict = STalign.download_aba_ontology(url, 'allen_ontology.csv')

In [ ]:
imageurl = 'http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/ara_nissl/ara_nissl_50.nrrd'
labelurl = 'http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/annotation_50.nrrd'
imagefile, labelfile = STalign.download_aba_image_labels(imageurl, labelurl, 'aba_nissl.nrrd', 'aba_annotation.nrrd')

In [ ]:
# Rasterize
X_, Y_, W = rasterize(x, y, dx=dx, blur=blur)

## Initialize alignment

In [ ]:
slice_idx = 140

vol, hdr = nrrd.read(imagefile)
A = vol
vol, hdr = nrrd.read(labelfile)
L = vol

dxA = np.diag(hdr['space directions'])
nxA = A.shape
xA = [np.arange(n)*d - (n-1)*d/2.0 for n, d in zip(nxA, dxA)]

In [ ]:
theta_deg = 90

points_atlas = np.array([[0, 0]])
points_target = np.array([[-3700, 0]])
Li, Ti = STalign.L_T_from_points(points_atlas, points_target)

xJ = [Y_, X_]
J = W[None] / np.mean(np.abs(W))
xI = xA
I = A[None] / np.mean(np.abs(A), keepdims=True)
I = np.concatenate((I, (I - np.mean(I))**2))
Inorm = STalign.normalize(I, t_min=0, t_max=1)
Jnorm = STalign.normalize(J, t_min=0, t_max=1)

In [ ]:
sigmaA = 0.1
sigmaB = 0.1
sigmaM = 0.1
muA = torch.tensor([0.7, 0.7, 0.7], device='cpu')
muB = torch.tensor([0, 0, 0], device='cpu')

scale_x = 4
scale_y = 4
scale_z = 0.9
theta0 = (np.pi/180) * theta_deg

T = np.array([-xI[0][slice_idx], np.mean(xJ[0])-(Ti[0]*scale_y), np.mean(xJ[1])-(Ti[1]*scale_x)])
scale_atlas = np.array([[scale_z, 0, 0], [0, scale_x, 0], [0, 0, scale_y]])
L_mat = np.array([[1.0, 0.0, 0.0], [0.0, np.cos(theta0), -np.sin(theta0)], [0.0, np.sin(theta0), np.cos(theta0)]])
L_mat = np.matmul(L_mat, scale_atlas)

## Run 3D-to-slice alignment

In [ ]:
%%time
transform = STalign.LDDMM_3D_to_slice(
    xI, Inorm, xJ, Jnorm,
    T=T, L=L_mat,
    nt=4, niter=800,
    a=250,
    device='cpu',
    sigmaA=sigmaA, sigmaB=sigmaB, sigmaM=sigmaM,
    muA=muA, muB=muB,
)

In [ ]:
A_out = transform['A']
v = transform['v']
xv = transform['xv']
Xs = transform['Xs']

## Analyze results

In [ ]:
df_result = STalign.analyze3Dalign(
    labelfile, xv, v, A_out, xJ, dx,
    scale_x=scale_x, scale_y=scale_y,
    x=x, y=y, X_=X_, Y_=Y_,
    namesdict=namesdict, device='cpu',
)
df_result

In [ ]:
STalign.plot_brain_regions(df_result)

In [ ]:
brain_regions = ['CA1']
STalign.plot_subset_brain_regions(df_result, brain_regions)